
A notebook with scripts to convert BFCL-formatted datasets into input datasets (with "messages", "tools" serving as inputs and "tool_calls" as ground truths) of OpenAI compatible format supported by custom evaluation with tool calling metric.


In [1]:
import copy
import re


def convert_to_tool(functions, mapping):
    functions = copy.deepcopy(functions)
    oai_tool = []
    for item in functions:
        if "." in item["name"]:
            # OAI does not support "." in the function name so we replace it with "_". ^[a-zA-Z0-9_-]{1,64}$ is the regex for the name.
            item["name"] = re.sub(r"\.", "_", item["name"])

        item["parameters"]["type"] = "object"
        item["parameters"]["properties"] = _cast_to_openai_type(item["parameters"]["properties"], mapping)

        oai_tool.append({"type": "function", "function": item})

    return oai_tool


def _cast_to_openai_type(properties, mapping):
    for key, value in properties.items():
        if "type" not in value:
            properties[key]["type"] = "string"
        else:
            var_type = value["type"]
            if mapping == GORILLA_TO_OPENAPI and var_type == "float":
                properties[key]["format"] = "float"
                properties[key]["description"] += " This is a float type value."
            if var_type in mapping:
                properties[key]["type"] = mapping[var_type]
            else:
                properties[key]["type"] = "string"

        # Currently support:
        # - list of any
        # - list of list of any
        # - list of dict
        # - list of list of dict
        # - dict of any

        if properties[key]["type"] == "array" or properties[key]["type"] == "object":
            if "properties" in properties[key]:
                properties[key]["properties"] = _cast_to_openai_type(properties[key]["properties"], mapping)
            elif "items" in properties[key]:
                properties[key]["items"]["type"] = mapping[properties[key]["items"]["type"]]
                if properties[key]["items"]["type"] == "array" and "items" in properties[key]["items"]:
                    properties[key]["items"]["items"]["type"] = mapping[properties[key]["items"]["items"]["type"]]
                elif properties[key]["items"]["type"] == "object" and "properties" in properties[key]["items"]:
                    properties[key]["items"]["properties"] = _cast_to_openai_type(
                        properties[key]["items"]["properties"], mapping
                    )
    return properties


GORILLA_TO_OPENAPI = {
    "integer": "integer",
    "number": "number",
    "float": "number",
    "string": "string",
    "boolean": "boolean",
    "bool": "boolean",
    "array": "array",
    "list": "array",
    "dict": "object",
    "object": "object",
    "tuple": "array",
    "any": "string",
    "byte": "integer",
    "short": "integer",
    "long": "integer",
    "double": "number",
    "char": "string",
    "ArrayList": "array",
    "Array": "array",
    "HashMap": "object",
    "Hashtable": "object",
    "Queue": "array",
    "Stack": "array",
    "Any": "string",
    "String": "string",
    "Bigint": "integer",
}

In [2]:
import json


def read_jsonl(file_path):
    """Reads a JSONL file and returns a list of JSON objects."""
    data = []
    with open(file_path, "r", encoding="utf-8") as file:
        for line in file:
            try:
                json_object = json.loads(line.strip())
                data.append(json_object)
            except json.JSONDecodeError as e:
                print(f"Error parsing line: {line.strip()} - {e}")
    return data

In [3]:
def write_jsonl(file_path, data):
    with open(file_path, "w") as file:
        for entry in data:
            json.dump(entry, file, ensure_ascii=False)
            file.write("\n")

In [4]:
def convert_bfcl_to_custom_tool_calling(input_file, gt_file, output_file):
    bfcl_simple_inputs = read_jsonl(input_file)
    bfcl_simple_gts = read_jsonl(gt_file)

    result = []

    for i in range(len(bfcl_simple_inputs)):
        record_output = {}
        record_input = bfcl_simple_inputs[i]
        record_gt = bfcl_simple_gts[i]

        record_output["messages"] = record_input["question"][0]
        record_output["tools"] = convert_to_tool(record_input["function"], GORILLA_TO_OPENAPI)

        tool_calls = []
        for gt in record_gt["ground_truth"]:
            gt_fn_name = next(iter(gt))
            gt_fn_args = gt[gt_fn_name]

            fn_name = gt_fn_name.replace(".", "_")  # Replace dots in function names
            fn_args = {}
            for arg_name, arg_value in gt_fn_args.items():
                corrected_arg_value = arg_value

                # The following lines correct some deficiencies in original datasets where arguments were put into (nested) lists

                # if isinstance(arg_value, list):
                #     filtered_args = [arg for arg in arg_value if arg != ""] # Filter out empty strings

                #     if len(filtered_args) == 1:
                #         corrected_arg_value = filtered_args[0] # Flatten
                #     else:
                #         corrected_arg_value = filtered_args

                fn_args[arg_name] = corrected_arg_value

            fn = {
                "function": {
                    "name": fn_name,
                    "arguments": fn_args,
                }
            }
            tool_calls.append(fn)

        record_output["tool_calls"] = tool_calls

        result.append(record_output)

    write_jsonl(output_file, result)

In [7]:
convert_bfcl_to_custom_tool_calling(
    "~/workspace/bfcl/BFCL_v3_simple.json",
    "~/workspace/bfcl/BFCL_v3_simple_gt.json",
    "~/workspace/bfcl/BFCL_v3_simple_sample.jsonl",
)

FileNotFoundError: [Errno 2] No such file or directory: '~/workspace/bfcl/BFCL_v3_simple.json'

In [6]:
convert_bfcl_to_custom_tool_calling(
    "~/workspace/bfcl/BFCL_v3_parallel_multiple.json",
    "~/workspace/bfcl/BFCL_v3_parallel_multiple_gt.json",
    "~/workspace/bfcl/BFCL_v3_parallel_multiple_sample.jsonl",
)